# Market Intelligence Lab (MIL) 2.0 — Phase 7: Alpha Research Track 1
## NSE Institutional Delivery Flow & PIT Post-Earnings Announcement Drift (PEAD)

**Objective:**
Investigate whether combining:
1. Microstructure Accumulation (`deliv_zscore_60d`, `clv_1d`, `deliv_clv_impulse`, `atpt_zscore_60d`)
2. Point-in-Time Institutional Deltas (`fii_dii_delta_qoq`, `promoter_pledge_delta`)
3. Point-in-Time Earnings Surprise (`sue_eps`, `opm_surprise_bps`, `pead_reaction_score` with strict PIT observation gating)

improves out-of-sample risk-adjusted returns (CPCV Sharpe) over the 0.4322 deterministic baseline net of Angel One retail delivery friction.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import math

# Alpha v2 Feature Set Definition
BASELINE_FEATURES = [
    "tactical_20d", "trend_60d", "structural_200d", "rsi_regime",
    "low_vol_quality", "volume_accumulation", "relative_strength_nifty", "news_sentiment_7d"
]

FLOW_FEATURES = [
    "deliv_zscore_60d", "clv_1d", "deliv_clv_impulse", "atpt_zscore_60d",
    "fii_dii_delta_qoq", "promoter_pledge_delta"
]

PEAD_FEATURES = [
    "sue_eps", "opm_surprise_bps", "is_observed_pead", "pead_reaction_score"
]

ALPHA_V2_FEATURES = BASELINE_FEATURES + FLOW_FEATURES + PEAD_FEATURES
print(f"Total Alpha v2 Features: {len(ALPHA_V2_FEATURES)}")

In [ ]:
# Statistical Evaluation: Information Coefficient (Spearman Rank IC) vs Forward 20D Net Return
def evaluate_feature_information_coefficients(df, feature_cols, target_col="net_return"):
    records = []
    for feat in feature_cols:
        valid = df[[feat, target_col]].dropna()
        if len(valid) < 30:
            continue
        corr, pval = stats.spearmanr(valid[feat], valid[target_col])
        t_stat = corr * math.sqrt(len(valid) - 2) / max(math.sqrt(1 - corr**2), 1e-6)
        records.append({
            "feature": feat,
            "rank_ic": round(float(corr), 4),
            "p_value": round(float(pval), 5),
            "t_stat": round(float(t_stat), 2),
            "significant_5pct": bool(pval < 0.05)
        })
    res_df = pd.DataFrame(records).sort_values("rank_ic", ascending=False)
    return res_df

print("Rank IC evaluator defined.")

In [ ]:
# Synthetic / Historical Calibration Run: Verify Feature Independence & Information Bounds
np.random.seed(42)
n_samples = 2500

synthetic_data = {
    "tactical_20d": np.random.uniform(0.1, 0.9, n_samples),
    "trend_60d": np.random.uniform(0.1, 0.9, n_samples),
    "structural_200d": np.random.uniform(0.1, 0.9, n_samples),
    "rsi_regime": np.random.uniform(0.1, 0.9, n_samples),
    "low_vol_quality": np.random.uniform(0.1, 0.9, n_samples),
    "volume_accumulation": np.random.uniform(0.1, 0.9, n_samples),
    "relative_strength_nifty": np.random.uniform(0.1, 0.9, n_samples),
    "news_sentiment_7d": np.random.uniform(0.1, 0.9, n_samples),
    # Institutional Flow
    "deliv_zscore_60d": np.random.normal(0.0, 1.0, n_samples).clip(-4.0, 4.0),
    "clv_1d": np.random.uniform(0.0, 1.0, n_samples),
    "atpt_zscore_60d": np.random.normal(0.0, 1.0, n_samples).clip(-4.0, 4.0),
    "fii_dii_delta_qoq": np.random.normal(0.0, 1.5, n_samples),
    "promoter_pledge_delta": np.random.normal(0.0, 0.5, n_samples),
    # PEAD
    "sue_eps": np.random.normal(0.0, 1.2, n_samples).clip(-5.0, 5.0),
    "opm_surprise_bps": np.random.normal(0.0, 150.0, n_samples).clip(-5000.0, 5000.0),
    "is_observed_pead": np.random.binomial(1, 0.75, n_samples),
}
synthetic_df = pd.DataFrame(synthetic_data)
synthetic_df["deliv_clv_impulse"] = (synthetic_df["deliv_zscore_60d"] * (2.0 * synthetic_df["clv_1d"] - 1.0)).clip(-8.0, 8.0)
synthetic_df["pead_reaction_score"] = np.where(
    synthetic_df["is_observed_pead"] == 1,
    synthetic_df["sue_eps"].clip(-3.0, 3.0) * np.exp(-np.random.uniform(0, 20, n_samples) / 10.0),
    0.0
)

# Simulated net return: Signal + Noise - 70 bps Angel One friction
true_signal = (
    0.025 * synthetic_df["tactical_20d"] +
    0.020 * synthetic_df["deliv_clv_impulse"] +
    0.022 * synthetic_df["pead_reaction_score"] +
    0.015 * synthetic_df["low_vol_quality"]
)
friction_drag = 0.0070  # 70 bps roundtrip friction on ₹5,000 ticket
synthetic_df["net_return"] = true_signal + np.random.normal(0.0, 0.06, n_samples) - friction_drag

ic_results = evaluate_feature_information_coefficients(synthetic_df, ALPHA_V2_FEATURES, target_col="net_return")
print(ic_results.to_string(index=False))